# Week 3, day 2 — Extra practice 07 SOLUTIONS: transforming values   (L05)

Executed in the lab image (pandas 3.0.5) against the real files in `../data/`.
Every quoted number is what it actually printed.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 07 — Transforming values. Run this once.
import numpy as np
import pandas as pd

cust = pd.read_csv("../data/customers_messy.csv")
orders = pd.read_csv("../data/orders_long.csv")

print("customers:", cust.shape, "| orders:", orders.shape)
print("regions in customers:", sorted(cust["Region"].unique()))

### Question 1

**`Prarie`** appears in both files — `71` customers and `216` orders.

The same misspelling in both, which is at least consistent. It comes from
one source system, so both extracts inherited it.

Consistent and wrong is much safer than inconsistent — the join still works.
Q3 is what happens when you break that.

In [ ]:
print("customers:")
print(cust["Region"].value_counts().to_string())
print()
print("orders:")
print(orders["Region"].value_counts().to_string())

### Question 2

After replacing in both, `0` rows carry the old spelling and the corrected one has `71` customers and `216` orders.

The counts carried over exactly, which is the check that `replace` swapped
rather than dropped.

In [ ]:
c = cust["Region"].replace({"Prarie": "Prairie"})
o = orders["Region"].replace({"Prarie": "Prairie"})
print("customers with the old spelling:", (c == "Prarie").sum())
print("orders with the old spelling:   ", (o == "Prarie").sum())
print()
print("customers with the fixed spelling:", (c == "Prairie").sum())
print("orders with the fixed spelling:   ", (o == "Prairie").sum())

### Question 3

Shared regions go from **`8`** to **`7`** when only one side is fixed. -> `Prarie` is lost, and **`216` orders** would no longer match.

Fixing the spelling in one file and not the other is worse than leaving it
alone. Before, both said `Prarie` and joined fine. After, one says `Prairie`
and the other `Prarie`, and 216 orders match nothing.

An inner join would silently drop them. A left join would give them `NaN`
customer details. Neither raises.

When you standardise a key, standardise it everywhere in the same commit —
or better, do it once at the boundary where the data enters, so no consumer
ever sees the raw spelling.

In [ ]:
fixed_cust = cust.copy()
fixed_cust["Region"] = fixed_cust["Region"].replace({"Prarie": "Prairie"})

before = set(cust["Region"]) & set(orders["Region"])
after = set(fixed_cust["Region"]) & set(orders["Region"])
print("shared regions before the fix:", len(before))
print("shared regions after fixing ONE side:", len(after))
print("lost:", sorted(before - after))
print()
print("orders that would no longer match:",
      (orders["Region"] == "Prarie").sum())

### Question 4

`['Consumer', 'Corporate', 'Home Office', 'Small Business']` -> `['consumer', 'corporate', 'home_office', 'small_business']`.

Two chained `.str` methods, vectorised, no `map` or `apply` anywhere.

`.str` is the right tool for anything that is a transformation of the text
itself. Reserve `map` for lookups and `apply` for logic that needs more than
one column.

In [ ]:
print("before:", sorted(cust["Segment"].unique()))
tidy = cust["Segment"].str.lower().str.replace(" ", "_", regex=False)
print("after: ", sorted(tidy.unique()))

### Question 5

Total profit `218818.97` -> **`310283.66`** after flooring losses at zero. -> `468` rows changed, a difference of `91464.69`.

Flooring the losses inflated reported profit by 42%.

468 of 1,093 orders lose money — 43% of the file — and the operation
asserts that none of them did. Nothing raised, no row count changed, and the
headline number moved by ninety-one thousand.

There are legitimate reasons to clip a column (a sensor that cannot read
below zero, a quantity that cannot be negative). 'The losses look untidy' is
not one of them.

In [ ]:
work = orders.copy()
before = work["Profit"].sum()
work.loc[work["Profit"] < 0, "Profit"] = 0
after = work["Profit"].sum()
print("total profit before: %.2f" % before)
print("total profit after:  %.2f" % after)
print("difference:          %.2f" % (after - before))
print("rows changed:", (orders["Profit"] < 0).sum())

# Flooring losses at zero asserts the business never loses money on an
# order. It inflates total profit by the entire value of the losses.

### Question 6

`standard 542`, `watch 455`, `key 96`, totalling `1093`. -> **`13`** orders satisfy both conditions.

The three counts add to the row count, so no case was missed.

The 13 overlapping orders — over 5,000 *and* loss-making — are the
interesting ones. They are big and they lost money, which is arguably the
most important category in the table, and this scheme gives them whichever
label was applied last.

In [ ]:
work = orders.copy()
work["Tier"] = "standard"
work.loc[work["Profit"] < 0, "Tier"] = "watch"
work.loc[work["Sales"] > 5000, "Tier"] = "key"
counts = work["Tier"].value_counts()
print(counts.to_string())
print()
print("total:", counts.sum(), "of", len(work))
print()
print("orders over 5000 AND loss-making:",
      ((orders["Sales"] > 5000) & (orders["Profit"] < 0)).sum())

### Question 7

`watch` then `key` -> `watch 455`, `key 96`. `key` then `watch` -> **`watch 468`, `key 83`**. -> the `13` overlapping orders move.

Same three rules, same data, two different tables — decided entirely by the
order the `.loc` lines happen to appear in.

With overlapping conditions, **the last assignment wins**. That is not a bug
and it is not documented anywhere in your code: someone reordering two lines
for readability changes 13 rows.

The fixes are to make the conditions mutually exclusive by construction
(`& ~`), or to use `np.select`, which takes an explicit ordered list of
conditions and makes the precedence visible. Either way the priority should
be something you stated, not something the line order implies.

In [ ]:
def build(order):
    w = orders.copy()
    w["Tier"] = "standard"
    for rule in order:
        if rule == "watch":
            w.loc[w["Profit"] < 0, "Tier"] = "watch"
        else:
            w.loc[w["Sales"] > 5000, "Tier"] = "key"
    return w["Tier"].value_counts()

print("watch then key:")
print(build(["watch", "key"]).to_string())
print()
print("key then watch:")
print(build(["key", "watch"]).to_string())
print()
overlap = ((orders["Sales"] > 5000) & (orders["Profit"] < 0)).sum()
print("orders over 5000 AND loss-making:", overlap)

### Question 8

`.str.upper()` on a float column -> **raises** `AttributeError: Can only use .str accessor with string values, not floating`.

A clear message naming both what `.str` needs and what it got.

Compare with worksheet 13 of the week 3, day 1 class, where `.str.upper()` on a
column of **dicts** returned `NaN` for every row instead of raising. The
accessor raises on a numeric column and stays silent on an `object` column
of non-strings — so the loud case is the one where the dtype is honest.

In [ ]:
print("Sales dtype:", orders["Sales"].dtype)
print(orders["Sales"].str.upper())